# The release gate: evidence, thresholds and a decision

**Module 20 — Secure Deployment and Release Governance**

Everything in this course converges on one artefact: a gate that says yes or no, with reasons. This notebook
builds that gate from six checks, each returning evidence rather than a boolean, and a decision rule that is
itself reviewable.

The gate is deterministic and self-contained: measured values come from a synthetic evaluation, and the same
inputs always produce the same decision.

## Learning objectives

1. Define release criteria as measurable thresholds with intervals.
2. Collect evidence per criterion and record it, not just pass/fail.
3. Implement a gate that distinguishes blocking from advisory checks.
4. Produce a decision record a reviewer can audit without rerunning anything.
5. Explain why the gate belongs in CI, not in a slide deck.

## Background

A release gate is a function from evidence to decision. Three properties make it trustworthy:

* **Pre-declared thresholds.** Deciding the bar after seeing the score is not a gate.
* **Intervals, not point estimates.** A criterion "accuracy ≥ 0.90" with a 95 % interval of [0.84, 0.93] is not
  met, even though the point estimate is.
* **Blocking versus advisory.** A missing model card blocks; a stale dashboard does not. Mixing them either
  blocks everything or nothing.

The gate below implements all three and writes a record that travels with the artefact.

## Dependencies and installation

```bash
python -m pip install "numpy>=2.1" "pandas>=2.2"
```

Standard library plus NumPy and pandas. The evaluation numbers are synthesised deterministically so the notebook
runs anywhere; in a real pipeline they would come from the evaluation job.

In [1]:
import hashlib
import json
import math
from datetime import date

import numpy as np
import pandas as pd

SEED = 83
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 160)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — Criteria declared before measurement

Six criteria, each with a threshold, a type (blocking or advisory) and an evidence source. This table is the
policy; it should be reviewed by security, legal and product before the first release.

In [2]:
criteria = pd.DataFrame(
    [
        {"id": "C1", "name": "accuracy lower bound", "threshold": 0.90, "direction": "min",
         "kind": "blocking", "evidence": "evaluation report, 95% Wilson lower bound"},
        {"id": "C2", "name": "robust accuracy", "threshold": 0.50, "direction": "min",
         "kind": "blocking", "evidence": "PGD-20 at eps=0.2"},
        {"id": "C3", "name": "membership advantage", "threshold": 0.15, "direction": "max",
         "kind": "blocking", "evidence": "advantage at 5% FPR"},
        {"id": "C4", "name": "parity gap", "threshold": 0.10, "direction": "max",
         "kind": "blocking", "evidence": "demographic parity difference"},
        {"id": "C5", "name": "artefact format", "threshold": "safetensors", "direction": "exact",
         "kind": "blocking", "evidence": "bundle manifest"},
        {"id": "C6", "name": "monitoring dashboard", "threshold": "exists", "direction": "exact",
         "kind": "advisory", "evidence": "link and owner"},
    ]
)
criteria

,id,name,threshold,direction,kind,evidence
0,C1,accuracy lower bound,0.9,min,blocking,"evaluation report, 95% Wilson lower bound"
1,C2,robust accuracy,0.5,min,blocking,PGD-20 at eps=0.2
2,C3,membership advantage,0.15,max,blocking,advantage at 5% FPR
3,C4,parity gap,0.1,max,blocking,demographic parity difference
4,C5,artefact format,safetensors,exact,blocking,bundle manifest
5,C6,monitoring dashboard,exists,exact,advisory,link and owner


**Expected output** — six rows, each with a direction: `min` means the lower bound must clear the bar, `max`
means the upper bound must stay under it. Four numeric blocking criteria, one categorical blocking criterion
and one advisory criterion. The mix is deliberate: a gate that only checks numbers misses supply-chain and operational
readiness.

### Step 2 — Measure, with an interval where the quantity is estimated

Accuracy is estimated from 600 evaluation rows, so it gets a Wilson interval. Robust accuracy, advantage and the
parity gap are also estimates; the same discipline applies.

In [3]:
def wilson(successes, n, z=1.96):
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


n_eval = 600
correct = int(0.915 * n_eval)          # 549 correct: point estimate above the bar
acc_low, acc_high = wilson(correct, n_eval)

measurements = {
    "C1": {"value": round(correct / n_eval, 3), "low": round(acc_low, 3), "high": round(acc_high, 3)},
    "C2": {"value": 0.62, "low": 0.55, "high": 0.69},
    "C3": {"value": 0.08, "low": 0.06, "high": 0.11},
    "C4": {"value": 0.13, "low": 0.09, "high": 0.17},
    "C5": {"value": "safetensors", "low": None, "high": None},
    "C6": {"value": "exists", "low": None, "high": None},
}
print(f"accuracy point estimate {measurements['C1']['value']:.3f} | 95% interval "
      f"[{measurements['C1']['low']:.3f}, {measurements['C1']['high']:.3f}]")

accuracy point estimate 0.915 | 95% interval [0.890, 0.935]


**Expected output** — a point estimate of 0.915 with a lower bound near 0.89. The point estimate clears 0.90; the
lower bound does not. Under the pre-declared rule this criterion fails, and that is the correct outcome.

### Step 3 — The gate

Three outcomes per criterion: pass, fail (blocking), warn (advisory). Numeric criteria are judged on the
*conservative* bound, which is the lower bound for a "≥" criterion.

In [4]:
def evaluate_criterion(row, measurement):
    if isinstance(row.threshold, str):
        passed = measurement["value"] == row.threshold
        return ("pass" if passed else "fail"), f"{measurement['value']} (required {row.threshold})"
    if row.direction == "min":            # higher is better: judge the lower bound
        conservative = measurement["low"]
        passed = conservative >= row.threshold
        label = "lower bound"
    else:                                 # lower is better: judge the upper bound
        conservative = measurement["high"]
        passed = conservative <= row.threshold
        label = "upper bound"
    detail = f"{label} {conservative:.3f} vs {row.threshold:.2f}"
    if passed:
        return "pass", detail
    return ("fail" if row.kind == "blocking" else "warn"), detail


rows = []
for row in criteria.itertuples():
    status, detail = evaluate_criterion(row, measurements[row.id])
    rows.append({"id": row.id, "name": row.name, "kind": row.kind, "status": status, "detail": detail})
gate = pd.DataFrame(rows)
gate

,id,name,kind,status,detail
0,C1,accuracy lower bound,blocking,fail,lower bound 0.890 vs 0.90
1,C2,robust accuracy,blocking,pass,lower bound 0.550 vs 0.50
2,C3,membership advantage,blocking,pass,upper bound 0.110 vs 0.15
3,C4,parity gap,blocking,fail,upper bound 0.170 vs 0.10
4,C5,artefact format,blocking,pass,safetensors (required safetensors)
5,C6,monitoring dashboard,advisory,pass,exists (required exists)


**Expected output** — C1 and C4 fail as blocking; C2, C3, C5 and C6 pass. Two failures, both real: the accuracy
lower bound is below the bar, and the parity gap's *upper* bound exceeds it. C3 passes because its upper bound
(0.11) stays under the 0.15 ceiling — the opposite direction of comparison from C1, which is why the direction
column exists.

### Step 4 — The decision rule

The release decision is a function of blocking statuses only. Advisory warnings are recorded and ignored for the
decision, which must be stated explicitly so nobody relitigates it later.

In [5]:
blocking = gate[gate.kind == "blocking"]
advisory = gate[gate.kind == "advisory"]
decision = "GO" if (blocking.status != "fail").all() else "NO-GO"
print(f"decision: {decision}")
print(f"blocking failures: {list(blocking[blocking.status == 'fail'].id)}")
print(f"advisory warnings: {list(advisory[advisory.status != 'pass'].id)}")
assert decision == "NO-GO"

decision: NO-GO
blocking failures: ['C1', 'C4']
advisory warnings: []


**Expected output** — `decision: NO-GO`, blocking failures `['C1', 'C4']`, no advisory warnings. The assertion
pins the outcome: if a threshold is edited later, the notebook fails loudly rather than silently approving a
release.

### Step 5 — Bind the decision to an artefact

A decision without a digest is unverifiable: the artefact can change after approval. Hash the model card, the
config and the evidence table, and store the combined digest in the record.

In [6]:
record = {
    "release": "toy-classifier-1.1.0",
    "date": date(2026, 3, 9).isoformat(),
    "decision": decision,
    "criteria_version": "2026-02-28",
    "evidence": json.loads(gate.to_json(orient="records")),
}
digest = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()
print("record digest:", digest[:32])
print("criteria reviewed before measurement:", record["criteria_version"] < record["date"])

record digest: b2892740adf639bb71ddc81177f62ea4
criteria reviewed before measurement: True


**Expected output** — a 32-character prefix of the digest and `True`. The date comparison is a small but real
control: criteria must predate the decision they justify.

### Step 6 — What happens next

The gate returns NO-GO twice in most real programmes. The useful output is the remediation list with owners,
not the decision itself.

In [7]:
remediation = pd.DataFrame(
    [
        {"id": "C1", "action": "raise evaluation size to 2000 rows or improve the model",
         "owner": "ml", "due": "2026-03-20", "unblocks": "accuracy lower bound"},
        {"id": "C4", "action": "reduce parity gap below 0.10 or document an approved exception",
         "owner": "responsible-ai", "due": "2026-03-27", "unblocks": "fairness criterion"},
    ]
)
remediation

,id,action,owner,due,unblocks
0,C1,raise evaluation size to 2000 rows or improve ...,ml,2026-03-20,accuracy lower bound
1,C4,reduce parity gap below 0.10 or document an ap...,responsible-ai,2026-03-27,fairness criterion


**Expected output** — two rows, each mapping a failure to an action, an owner and an unblocking statement. This
table is what a release review actually needs.

## Exercises

1. **Make one criterion advisory.** Change C4 to advisory and re-run. What is the decision, and what argument would
   justify that change?
2. **Widen the evidence.** Increase `n_eval` to 2000 with the same 0.915 accuracy and recompute the lower bound.
   Does C1 pass?
3. **Add a seventh criterion.** Add "red-team high-severity findings = 0" as blocking, wire it to a measurement,
   and show the decision.
4. **Automate the record.** Write the record to a JSON file and compute its digest from the file bytes, not from
   the dict. Why does that difference matter for verification?

## Reflection questions

1. Why must the gate judge the conservative bound rather than the point estimate?
2. What is the failure mode of a gate whose criteria are all advisory?
3. Who should own the criteria table, and how often should it change?

## References

1. NIST, *AI RMF 1.0*, NIST AI 100-1, 2023 — govern, map, measure, manage.
2. NIST, *SP 800-218*, Secure Software Development Framework, 2022 — release gates.
3. ISO/IEC 42001:2023, AI management systems.
4. OWASP, *Top 10 for LLM Applications*, 2025 — deployment and monitoring.